**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/robotka-2023"], check=True)
    os.chdir("/content/encore/entries/robotka-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Robotka et al. (2023) from CRCNS aa-5

**Paper.** Robotka, Thomas, Yu, Wood, Elie, Gahr & Theunissen (2023), "Sparse ensemble neural code
for a complete vocal repertoire", Cell Reports 42, 112034, doi:10.1016/j.celrep.2023.112034.

**Dataset.** CRCNS aa-5, "Simultaneous extracellular recordings of avian auditory neurons in freely
behaving zebra finches...", doi:10.6080/K0TT4P5Q, read from the public bucket
`crcnsarchive/aa-5/live/`. CRCNS data sets are not versioned: the version is recorded as
`unversioned`, the one published version there is, and the MD5 sums of every file read (from the
data set's `checksums.md5`) are printed below so that a later change can be detected. The paper
cites this DOI and no version; the bucket holds a single copy (`live/`, data descriptor v0.6 of
29 Nov 2022), so that is what is pinned.

**Claims.** `claims/robotka-2023.yaml`, written from the version of record with its STAR Methods and
supplement, and the paper's code (github.com/ftheunissen/Ensemble-Code-Cell-Reports-2022 at commit
f2d2376). Method details the claims file lacks are taken from the paper and that code, cited in the
helper `aa5.py` and below.

| claim | status here |
|---|---|
| stimulus-set-link | tested |
| paper-good-units-link (link claim added by the author) | tested |
| auditory-units | tested |
| inhibited-units | tested |
| fig3c-peak-call-types | tested |
| fig4a-unit-z-vs-zt | tested |
| good-units-100 | tested |
| fig4b-ensemble-20 | tested |
| fig4b-top-ensembles | tested |
| fig4b-confusions | tested |
| figs4-per-bird-and-sex | tested; blocked when the selection gives a bird fewer than 15 good units |
| figs6-primary-vs-secondary | blocked: no brain area per unit in the dataset |
| fig5e-volume | tested |
| fig5f-si-unrelated | tested |
| fig5g-entropy-selectivity | tested |
| fig5h-neural-noise | tested |
| fig5-multiple-regression | tested |
| pcs-variance | tested |
| table1-snr | tested |
| figs5-region-anova | blocked: no brain area per unit in the dataset |
| fig6a-acoustic-clusters, acoustic-labels-decoding, fig6c-number-of-clusters, fig6c-ari-call-over-acoustic, figs10-ensemble-size | blocked: the vocalization database and acoustic labels are not public (the call-type half of fig6c-number-of-clusters is computed) |
| anesth-auditory-units, anesth-decoding-better, anesth-regressions, anesth-primary-secondary-rate | blocked: CRCNS aa-4 not analysed in this run |
| figs9-per-region-regressions | blocked: no brain area per unit (and aa-4 not analysed) |

Reading the data streams about 164 GB of tar.gz from the bucket (four sites at a time); this takes
15 to 30 minutes and is most of the run time.

In [ ]:
import time, resource, warnings
from itertools import combinations
from math import comb

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import aa5

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T_START = time.time()

## Data check

The bucket's listing and the MD5 sums of the files read, then every site is streamed once and each
unit reduced to spike times, the stored KDE, labels and the stimulus waveforms (aa5.read_all).

In [ ]:
listing = aa5.list_files()
md5 = aa5.checksums()
tars = sorted([(k, s) for k, s in listing if k.endswith(".tar.gz")], key=lambda x: -x[1])
print(len(listing), "objects;", len(tars), "site files,", round(sum(s for _, s in tars) / 1e9, 1), "GB")
for k, s in listing:
    if not k.endswith("/"):
        print(f"{s:>12d}  {md5.get(k.replace(aa5.PREFIX, ''), '-'):32s}  {k}")

In [ ]:
t0 = time.time()
units, waves, other = aa5.read_all([k for k, _ in tars], workers=4, log=lambda s: None)
print(f"streamed {len(tars)} sites in {(time.time() - t0) / 60:.1f} min; {len(units)} unit pickles; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
# What a unit pickle holds: the unit dictionary's keys and every column of both tables
print("unitInfo keys:", sorted({k for u in units for k in u["info_keys"]}))
print("dfAbsTime columns:", sorted({c for u in units for c in u["abs_columns"]}))
print("dfRelTime columns:", sorted({c for u in units for c in u["rel_columns"]}))
print("other tar members:", len(other), "all AppleDouble '._' files:", all(o[0].split('/')[-1].startswith('._') for o in other))

ut = aa5.unit_table(units)
print()
print("pickles per bird:", ut.groupby("bird").size().to_dict(), " sites per bird:", ut.groupby("bird").site.nunique().to_dict())
empty = ut[ut.n_trials_all == 0]
print("empty pickles (no stimulus rows):", len(empty), "in sites", empty.site.value_counts().to_dict())
print("pickles with vocalization (call-type) trials:", (ut.n_calls > 0).sum(), "; song-only:", ((ut.n_calls == 0) & (ut.n_trials_all > 0)).sum())
print("call types per unit (units with any):", ut[ut.n_calls > 0].n_calls.value_counts().sort_index().to_dict())
print(f"SpikeSNR range {ut.snr.min():.2f} to {ut.snr.max():.2f}; units with SNR >= 5: {(ut.snr >= 5).sum()}")

# consistency of the per-trial fields
bad_tr = sum(len(r["spikeTimes"]) != r["nTrials"] for u in units for r in u["rows"])
bad_mic = sum(r["micShape"][0] != r["nTrials"] for u in units for r in u["rows"] if len(r["micShape"]) == 2)
allsp = np.concatenate([s for u in units for r in u["rows"] for s in r["spikeTimes"] if len(s)])
print("rows where len(spikeTimes) != nTrials:", bad_tr, "; where micWav rows != nTrials:", bad_mic)
print(f"spike times relative to onset: {allsp.min():.3f} to {allsp.max():.3f} s (descriptor: -0.5 to 4.5 s); {len(allsp)} spikes")
kde_grid = pd.Series([(r["nKDE"], round(r["tKDE0"], 3), round(r["dtKDE"], 4)) for u in units for r in u["rows"]]).value_counts()
print("KDE time grids (n samples, start s, step s):", kde_grid.to_dict())
stim_grid = pd.Series([(round(r["tStim0"], 3), round(1 / r["dtStim"]), r["stimLen"]) for u in units for r in u["rows"]]).value_counts()
print("stimulus grids (start s, Hz, samples):", stim_grid.head(3).to_dict(), "...", len(stim_grid), "kinds")
nhash = {s: len({r["stimHash"] for u in units for r in u["rows"] if r["file"] == s}) for s in waves}
print("vocalization stimuli:", len(waves), "; with a waveform that differs between units:", sum(v > 1 for v in nhash.values()))
kbw = pd.Series([u["KDE_BW"] for u in units])
print(f"KDE_BW (descriptor: bandwidth in s of the coarse rate used for RateThreshold): {kbw.min()} to {kbw.max()},",
      "pickles per value:", kbw.value_counts().sort_index().to_dict())
rth = pd.Series([u["RateThreshold"] for u in units])
print(f"RateThreshold (descriptor: spikes/s): {rth.min()} to {rth.max()}, {rth.nunique()} distinct values")

# recordings deposited twice: identical spike trains in two pickles
nonempty = ut[ut.n_trials_all > 0]
dup = nonempty[nonempty.spike_hash.duplicated(keep=False)]
print("non-empty pickles with a spike train identical to another pickle:", len(dup))
print("sites per bird and depth:", ut.groupby("bird").site.apply(lambda s: sorted(set(x.split('_')[1] for x in s))).to_dict())

In [ ]:
# Stimulus set: call types and exemplar counts over all units
voc = pd.DataFrame([(s, w[0]["call_type"]) for s, w in waves.items()], columns=["stim", "call"])
print(voc.call.value_counts().to_dict(), "total", len(voc))

### Are the paper's units in the dataset?

decodingCallTypesNaiveBayesPart2.ipynb, in the paper's code at the pinned commit, prints the names of
its 100 good units (site and pickle name, with their single-unit PCC). Read here only to check which
of them the dataset holds; nothing else is taken from it.

In [ ]:
import re, requests
CODE_URL = ("https://raw.githubusercontent.com/ftheunissen/Ensemble-Code-Cell-Reports-2022/"
            "f2d237639217425e7fc81d694708f19428a3b026/decodingCallTypesNaiveBayesPart2.ipynb")
nbp = requests.get(CODE_URL, timeout=60).json()
txt = "".join("".join(o.get("text", [])) for c in nbp["cells"] if c["cell_type"] == "code" for o in c.get("outputs", []))
paper_good = dict(re.findall(r"(ZF[\w.]+?_goodPlayback-e\d+-c\d+\.pkl) ([\d.]+)", txt))
uids = set(ut.uid)
in_ds = [n for n in paper_good if n in uids]
renamed = [n for n in paper_good if n not in uids and n.replace(".old", "") in uids]
missing = [n for n in paper_good if n not in uids]
print(f"paper's good units listed: {len(paper_good)}; in the dataset under the same name: {len(in_ds)}; missing: {len(missing)}")
print("missing, by site:", pd.Series([n.split("_goodPlayback")[0] for n in missing]).value_counts().to_dict())
print("of these, a pickle of the same name exists in the site without '.old':", len(renamed))
print("dataset site ZF4F_2t_190613_101015 holds", int((ut.site == "ZF4F_2t_190613_101015").sum()), "pickles")

### The per-trial database and the clock of spikes and stimuli

Auditory units (SNR >= 5, auditory test p <= 0.01, with vocalization trials; C1 and
GenerateDataBase.ipynb), their trial Z (C2), the PCA of trial-averaged KDE responses and the
per-trial T1 to T5 (C4). Spike times are stored relative to stimulus onset. The check that they share
the stimulus clock: the population rate around onset, and decoding call type from spike counts in a
window before onset (where no information should be) against the same window after onset.

In [ ]:
t0 = time.time()
db = aa5.build_database(units, waves)
aud = db["aud"]
print(f"database built in {time.time() - t0:.0f} s: {len(aud)} auditory units, {db['n_kde_rows']} KDE rows in the PCA")
print("auditory units per bird:", aud.groupby("bird").size().to_dict())
print("of the paper's good units held by the dataset,", sum(n in set(aud.uid) for n in in_ds), "of", len(in_ds), "are auditory here")
print("  and", sum((db["feat"][n][1] > 0).sum() == len(db["stims"]) for n in in_ds if n in db["feat"]), "of them have all 110 stimuli")
byuid = {u["Site"] + "_" + u["pkl"]: u for u in units}

# population PSTH of auditory units, vocalization trials
bins = np.arange(-0.5, 1.0001, 0.01)
h = np.zeros(len(bins) - 1); ntr = 0
for x in aud.uid:
    for r in aa5.voc_rows(byuid[x]):
        for s in r["spikeTimes"]:
            h += np.histogram(s, bins)[0]; ntr += 1
rate = h / ntr / 0.01
pre, post = rate[bins[:-1] < 0].mean(), rate[(bins[:-1] >= 0.02) & (bins[:-1] < 0.1)].mean()
first = bins[:-1][np.argmax(rate > pre + 5 * rate[bins[:-1] < 0].std())]
print(f"mean rate before onset {pre:.2f} spikes/s, 20 to 100 ms after onset {post:.2f} spikes/s; first 10 ms bin above baseline + 5 SD starts at {first * 1000:.0f} ms")
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(bins[:-1] * 1000, rate); ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("time from stimulus onset (ms)"); ax.set_ylabel("spikes/s"); ax.set_title(f"{len(aud)} auditory units, range {rate.min():.1f} to {rate.max():.1f}")
plt.show()

**Result: partially reproduced (paper-good-units-link).** 91 of the 100 good units the paper's code
lists are in the dataset under the same site and pickle name, and all 91 are auditory with all 110
stimuli here. The other 9 come from a site folder named ZF4F_2t_190613_101015.old; the dataset's
site ZF4F_2t_190613_101015 holds 18 pickles, none with those names (they are listed, with their
electrodes, in the good-units-100 section). The 91 held by the dataset are 34 of ZF4F, 19 of ZF5M,
16 of ZF6M and 22 of ZF7F.

In [ ]:
# Clock check by decoding: single-unit count features from [-0.5, 0) against [0, 0.5) s
rng = np.random.default_rng(101)
full_all = [x for x in aud.uid if (db["feat"][x][1] > 0).sum() == len(db["stims"])]
def count_feats(x, lo, hi):
    tr = db["trials"][x]
    u = byuid[x]
    cnt = np.array([np.sum((s >= lo) & (s < hi)) for r in aa5.voc_rows(u) for s in r["spikeTimes"]], float)
    F, C = aa5.unit_features(tr.assign(z=cnt), np.zeros((len(tr), 5)), db["stims"])
    return F, C
out = {}
for name, (lo, hi) in {"before onset [-0.5, 0)": (-0.5, 0.0), "after onset [0, 0.5)": (0.0, 0.5)}.items():
    p = []
    for x in full_all:
        F, C = count_feats(x, lo, hi)
        conf, nt = aa5.decode(F[None], C[None], db["y"], 10, rng, code="Z", cv="trial", nperm=10)
        p.append(aa5.pcc_code(conf, nt))
    out[name] = np.array(p)
    print(f"{name}: mean single-unit PCC {np.mean(p):.2f}% over {len(p)} units (chance 10%), t vs 10 = {stats.ttest_1samp(p, 10).statistic:.2f}")

In [ ]:
# The stored spikeKDE against a Gaussian KDE of the spike times (STAR Methods, PC analysis: "Gaussian
# kernels of 30 ms"), on 3 stimuli of each of 60 random auditory units (aa5.kde_check)
rng = np.random.default_rng(102)
recs = []
for x in rng.choice(list(aud.uid), 60, replace=False):
    for r in [r for r in aa5.voc_rows(byuid[x]) if r["nTrials"] >= aa5.MIN_TRIALS_KDE][:3]:
        if sum(len(s) for s in r["spikeTimes"]) >= 10:
            kq = aa5.kde_check(r)
            recs.append(dict(nTrials=r["nTrials"], scale=kq.pop("scale"), **{f"r_{round(k * 1000)}ms": v for k, v in kq.items()}))
kc = pd.DataFrame(recs)
rcols = [c for c in kc if c.startswith("r_")]
print(f"{len(kc)} stimulus rows; median correlation with the stored KDE by kernel SD:", kc[rcols].median().round(4).to_dict())
print("kernel SD of highest correlation, rows per SD:", kc[rcols].idxmax(axis=1).value_counts().to_dict())
print(f"30 ms: correlation above 0.99 in {(kc.r_30ms > 0.99).sum()} of {len(kc)} rows, range {kc.r_30ms.min():.3f} to {kc.r_30ms.max():.3f}")
print("stored KDE over the trial-averaged 30 ms KDE (scale), median by number of trials:", kc.groupby("nTrials").scale.median().round(3).to_dict())
# kde_matrix sets a row to zero when the stimulus has no spikes; the earlier test (sum of spike times == 0) would also have zeroed rows with spikes
n_old = sum(1 for x in aud.uid for r in aa5.voc_rows(byuid[x]) if r["nTrials"] >= aa5.MIN_TRIALS_KDE
            and sum(len(s) for s in r["spikeTimes"]) and not np.sum(np.concatenate(r["spikeTimes"])))
n_zero = sum(1 for x in aud.uid for r in aa5.voc_rows(byuid[x]) if r["nTrials"] >= aa5.MIN_TRIALS_KDE and not sum(len(s) for s in r["spikeTimes"]))
print(f"KDE rows of stimuli with no spikes (set to zero): {n_zero}; rows with spikes whose spike times sum to 0: {n_old}")

## stimulus-set-link

> The dataset holds the awake playback experiment the paper describes, four birds (ZF4F, ZF5M,
> ZF6M, ZF7F) and 110 distinct stimuli labelled with the 10 call types in the paper's numbers.

In [ ]:
paper_counts = dict(Be=12, LT=13, Th=5, Di=5, Ag=9, Wh=13, Ne=15, Te=15, DC=14, So=9)
got = voc.call.value_counts().to_dict()
tab = pd.DataFrame({"paper": paper_counts, "dataset": {k: got.get(k, 0) for k in paper_counts}})
tab["diff"] = tab.dataset - tab.paper
print(tab.T)
birds = sorted(ut.bird.unique())
print("birds:", birds, len(birds), "; distinct vocalization stimuli:", len(voc))
print("labels in the files not in the paper's list:", sorted(set(got) - set(paper_counts)))

**Result: reproduced.** The files hold 4 birds and 110 distinct vocalization stimuli whose call-type
counts equal the paper's for all 10 types (every difference is 0). The stimulus waveform of each
stimulus is the same in every unit that heard it.

## auditory-units

> Of 730 spike-sorted units in the awake dataset, 292 (40.1%) were auditory.

Auditory test as C1, on the spike-count difference over all trials of the pickle (song and
vocalizations; GenerateReport.ipynb calc_zscore_all), Methods reading (one-sample t-test, n - 1 df)
and the code's (n df). Denominators: (a) units with SNR above 5, (b) all units the dataset holds,
counted three ways: every pickle, every non-empty pickle, every pickle with vocalization trials (the
code's summary keeps only these: `if ncalls == 0: continue`).

In [ ]:
voc_u = ut[ut.n_calls > 0]
rows = []
for test in ["p_methods", "p_code"]:
    n_aud = int(((voc_u.snr >= 5) & (voc_u[test] <= 0.01)).sum())
    for name, den in [("(a) SNR above 5, with vocalization trials", int((voc_u.snr >= 5).sum())),
                      ("(a') SNR above 5, all pickles", int((ut.snr >= 5).sum())),
                      ("(b) all pickles", len(ut)),
                      ("(b) non-empty pickles", int((ut.n_trials_all > 0).sum())),
                      ("(b) pickles with vocalization trials", len(voc_u))]:
        rows.append(dict(test=test, denominator=name, total=den, auditory=n_aud, fraction=100 * n_aud / den))
res_aud = pd.DataFrame(rows)
print(res_aud.round(2).to_string(index=False))
print("auditory units among SNR >= 5 units with vocalization trials, by bird:",
      voc_u[(voc_u.snr >= 5) & (voc_u.p_methods <= 0.01)].groupby("bird").size().to_dict())
print("auditory test on all units regardless of SNR (vocalization pickles):", int((voc_u.p_methods <= 0.01).sum()))

**Result: reproduced.** 287 units are auditory under both the Methods' and the code's t-test (paper
292, tolerance 29.2). With the denominator read as the units the dataset holds with vocalization
trials, the total is 726 (paper 730) and the fraction 39.5% (paper 40.1%), both within tolerance;
every pickle (872) or every non-empty pickle (838) is outside the 730 +- 73 tolerance, and the SNR
reading (a) gives 414 units, a fraction of 69%.

## inhibited-units

> In the awake dataset 46 of the 292 auditory units (15.7%) were inhibited by sound.

In [ ]:
a = aud
n_inh = int((a.meanD < 0).sum())
print(f"inhibited (mean d < 0): {n_inh} of {len(a)} = {100 * n_inh / len(a):.1f}%  (paper 46 of 292 = 15.7%)")
print("by bird:", a[a.meanD < 0].groupby("bird").size().to_dict())

**Result: reproduced.** 47 of the 287 auditory units (16.4%) have a negative mean spike-count
difference, within the tolerances of 46 +- 4.6 and 15.7 +- 1.57%.

## table1-snr

> The awake units have a mean spike SNR of 7.42 (SD 2.52; quartiles 5.51, 6.50, 9.03).

In [ ]:
for name, s in [("(a) auditory units", aud.snr), ("(b) units with SNR above 5 and vocalization trials", voc_u.snr[voc_u.snr >= 5]),
                ("(b') all units with SNR above 5", ut.snr[ut.snr >= 5])]:
    q = s.quantile([0.25, 0.5, 0.75]).to_numpy()
    print(f"{name}: n = {len(s)}, mean {s.mean():.2f}, SD {s.std():.2f}, quartiles {q[0]:.2f}, {q[1]:.2f}, {q[2]:.2f}")
print("paper: mean 7.42 (tolerance 0.742), median 6.50 (tolerance 0.65)")

**Result: reproduced.** The auditory units have a mean SNR of 6.84 and a median of 6.07, both inside
the tolerances (6.68 to 8.16 and 5.85 to 7.15); under reading (b) the mean is 6.97 and the median
6.20. In every reading both values sit below the paper's, and the upper quartile (7.6 to 7.9) is
well below its 9.03.

## pcs-variance

> The first five principal components of the time-varying responses explain about 90% of their variance.

In [ ]:
evr = db["pca"].explained_variance_ratio_
print("explained variance ratio, PCs 1 to 10:", np.round(evr, 4))
print(f"cumulative, first 5 PCs: {evr[:5].sum():.4f}  (paper about 0.90, tolerance 0.09; the code printed 0.922)")
print("KDE rows (unit x stimulus with >= 5 trials):", db["n_kde_rows"])

**Result: reproduced.** The first five PCs explain 92.4% of the variance of the trial-averaged
responses (the code printed 92.2%).

## fig3c-peak-call-types

> Across the 292 auditory units, distress calls had the highest average response strength followed by
> song; 51 units had their peak response to distress calls and 61 to song; the preference is biased
> toward the longer calls (Ag, Be, So, Di).

zCT under the two readings of C3: (a) mean trial Z per call type; (b) the code's mean(d)/sd(d).

In [ ]:
za_list, zb_list = [], []
for x in aud.uid:
    za, zb = aa5.zct_unit(byuid[x], db["trials"][x])
    za_list.append(za); zb_list.append(zb)
ZA = pd.DataFrame(za_list, index=aud.uid)[aa5.CALL_TYPES]
ZB = pd.DataFrame(zb_list, index=aud.uid)[aa5.CALL_TYPES]
fig3 = {}
for name, Z in [("(a) mean trial Z", ZA), ("(b) code zCT", ZB)]:
    m = Z.mean().sort_values(ascending=False)
    peaks = Z.idxmax(axis=1).value_counts().reindex(aa5.CALL_TYPES, fill_value=0).sort_values(ascending=False)
    fig3[name] = (m, peaks)
    print(name)
    print("  mean zCT by call type, ranked:", m.round(3).to_dict())
    print("  peak units by call type:", peaks.to_dict(), " units with missing types:", int(Z.isna().any(axis=1).sum()))
    print("  top five by peak units:", list(peaks.index[:5]), "; Ag, Be, So, Di among them:", sum(c in peaks.index[:5] for c in ["Ag", "Be", "So", "Di"]))
    for b, g in Z.groupby(aud.set_index("uid").bird):
        print(f"   {b}: n={len(g)}, top mean {g.mean().idxmax()}, peaks Di {int((g.idxmax(axis=1) == 'Di').sum())}, So {int((g.idxmax(axis=1) == 'So').sum())}")
print("paper: Di first, So second; peaks Di 51 (tolerance 5.1), So 61 (6.1)")

**Result: reproduced, under the code's zCT (reading b), adopted as an assumption.** With the code's zCT,
which is what Figure 3C plots (SummaryStatistics.ipynb, cells 11 and 15, reads `zcall` from
summarySel.pkl), Di has the highest mean (0.623) and So the second (0.564); 53 units peak on Di and 58
on So (paper 51 and 61, both within tolerance); the five call types with the most peak units are So,
Ag, Di, Be and Th, which include Ag, Be, So and Di. Under the Methods' reading (a), So ranks first
(0.591), Ag second and Di third, and the So peak count (71) is outside tolerance: the claim would not
be reproduced under that reading.

## fig4a-unit-z-vs-zt

> For single units decoded independently, adding the temporal pattern (Z + T) to the response strength
> (Z) raises the mean PCC from 10.78% to 12.3% (a 14% increase), over 191 units with data for all 110
> calls.

Units: auditory units with all 110 vocalization stimuli (code: `nstimsUnit == 110`). Readings of C5:
(a) the Methods, one exemplar per call type held out, PCC as the mean diagonal of the row-normalized
matrix; (b) the code, one random trial per stimulus held out, PCC as trace over test trials, with
one repetition (the single-unit notebook) and with 100.

In [ ]:
full = full_all
n_ge2 = sum((db["feat"][x][1] >= 2).all() for x in full)
print(f"auditory units with all 110 stimuli: {len(full)} (paper 191); of these with >= 2 trials of every stimulus: {n_ge2}")
single = {}
readings = {"(a) exemplar, PCC_a": ("exemplar", 1, "pcc_methods"), "(b) trial, 1 rep, PCC_b": ("trial", 1, "pcc_code"),
            "(b) trial, 100 reps, PCC_b": ("trial", 100, "pcc_code")}
seed = 4000
for name, (cv, nperm, m) in readings.items():
    for cd in ["Z", "ZT"]:
        seed += 1
        single[(name, cd)] = aa5.run_ensembles(db, [[x] for x in full], code=cd, cv=cv, nperm=nperm, seed=seed)
rows = []
for name, (cv, nperm, m) in readings.items():
    z, zt = single[(name, "Z")][m], single[(name, "ZT")][m]
    t = stats.ttest_rel(zt, z)
    rows.append(dict(reading=name, n=len(z), PCC_Z=z.mean(), PCC_ZT=zt.mean(), increase_pct=100 * (zt.mean() / z.mean() - 1),
                     t=t.statistic, df=len(z) - 1, p=t.pvalue))
fig4a = pd.DataFrame(rows)
print(fig4a.round(4).to_string(index=False))
print("paper: Z 10.78 (tol 1.078), Z+T 12.3 (tol 1.23), t(190) = 10.06, p < 1e-5")
bird_of = aud.set_index("uid").bird
for name, (cv, nperm, m) in readings.items():
    d = pd.DataFrame({"Z": single[(name, "Z")][m], "ZT": single[(name, "ZT")][m]}, index=full)
    print(name, "by bird:", d.groupby(bird_of.loc[full].values).mean().round(2).to_dict("index"))

In [ ]:
# Variant: the Methods' unit rule, all 110 stimuli with at least 2 trials of each. Each unit is decoded alone,
# so these are the same decodings restricted to those units.
full2 = [j for j, x in enumerate(full) if (db["feat"][x][1] >= 2).all()]
rows = []
for name, (cv, nperm, m) in readings.items():
    z, zt = single[(name, "Z")][m][full2], single[(name, "ZT")][m][full2]
    t = stats.ttest_rel(zt, z)
    rows.append(dict(reading=name, n=len(z), PCC_Z=z.mean(), PCC_ZT=zt.mean(), increase_pct=100 * (zt.mean() / z.mean() - 1),
                     t=t.statistic, df=len(z) - 1, p=t.pvalue))
print("units with >= 2 trials of every stimulus (Methods):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** 187 units have all 110 stimuli (paper 191). Under every reading Z+T beats Z
by a paired t-test with p far below 1e-5, and both means are within tolerance: Methods reading 10.7%
and 12.0%, code reading 10.8% and 12.2 to 12.3%. With the Methods' unit rule (at least 2 trials of
every stimulus, 125 units) the means are 10.7 to 10.8% and 12.3 to 12.5%, also within tolerance, with
t(124) = 6.84 to 10.36; the 187 of the code's rule are closer to the paper's 191.

## good-units-100

> Exactly 100 units had a single-unit PCC Z+T above 12%, and they came from all four birds (ZF4F 43,
> ZF5M 19, ZF6M 16, ZF7F 22).

The count comes from one run, as the code; the range over 10 runs with other seeds is shown. The
good-unit set of each reading feeds the ensemble analyses of the same reading.

In [ ]:
bird_n = lambda us: pd.Series([bird_of.loc[x] for x in us]).value_counts().reindex(sorted(set(bird_of)), fill_value=0).to_dict()
good = {}
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    cv, nperm, m = readings[name]
    p = single[(name, "ZT")][m]
    good[name] = [x for x, v in zip(full, p) if v > 12]
    reps, rep_birds = [], []
    for s in range(10):
        r = aa5.run_ensembles(db, [[x] for x in full], code="ZT", cv=cv, nperm=1, seed=5000 + s)
        g_s = [x for x, v in zip(full, r[m]) if v > 12]
        reps.append(len(g_s)); rep_birds.append(bird_n(g_s))
    print(f"{name}: {len(good[name])} good units; by bird {bird_n(good[name])}; "
          f"over 10 other seeds {min(reps)} to {max(reps)} (mean {np.mean(reps):.1f}; within 90 to 110: {sum(90 <= r <= 110 for r in reps)} of 10)")
    rb = pd.DataFrame(rep_birds, index=[f"seed {5000 + s}" for s in range(10)]); rb["total"] = reps
    print(rb.T.to_string())
    print(f"   of the {len(in_ds)} paper good units in the dataset, {sum(n in set(good[name]) for n in in_ds)} are good here; "
          f"by bird {bird_n([n for n in in_ds if n in set(good[name])])}")
p100 = single[("(b) trial, 100 reps, PCC_b", "ZT")]["pcc_code"]
g100 = [x for x, v in zip(full, p100) if v > 12]
print("(b) with 100 repetitions:", len(g100), "good units; by bird", bird_n(g100))
print("the paper's 100 listed good units by bird:", pd.Series([n.split("_")[0] for n in paper_good]).value_counts().sort_index().to_dict(),
      "; the 91 the dataset holds:", bird_n(in_ds))
print("paper: 100 (tol 10); ZF4F 43 (4.3), ZF5M 19 (1.9), ZF6M 16 (1.6), ZF7F 22 (2.2)")

In [ ]:
# The 9 listed units of ZF4F_2t_190613_101015.old, with the single-unit PCC the code printed, and the 18 pickles the
# dataset holds for ZF4F_2t_190613_101015, with this attempt's single-unit PCC Z+T (code reading, one repetition).
# Matching them by spike train is not possible: the .old sorting is not deposited.
print("listed, missing:", {n.split("_goodPlayback-")[1]: float(paper_good[n]) for n in missing})
pb = dict(zip(full, single[("(b) trial, 1 rep, PCC_b", "ZT")]["pcc_code"]))
site18 = ut[ut.site == "ZF4F_2t_190613_101015"][["uid", "electrode", "snr", "p_code", "n_voc_stim"]].copy()
site18["auditory"] = site18.uid.isin(set(aud.uid)); site18["PCC_ZT_b"] = site18.uid.map(pb)
site18["uid"] = site18.uid.str.split("_goodPlayback-").str[1]
print(site18.round(3).to_string(index=False))
print("electrodes of the missing units:", sorted({n.split("-e")[1].split("-")[0] for n in missing}),
      "; of the dataset's pickles there:", sorted(set(site18.electrode)))

**Result: not reproduced.** One run of the code's reading gives 89 units with single-unit PCC Z+T above
12%, one below the tolerance (90 to 110); the Methods' reading gives 77. Every bird contributes units.
The count depends on the random held-out draw: over 10 other seeds the code's reading gives 87 to 101
(8 of 10 within tolerance) and 100 repetitions give 91; the Methods' reading gives 70 to 94 (1 of 10).
Per bird (code reading) ZF4F 39, ZF5M 20 and ZF7F 20 are within their tolerances and ZF6M 10 is not
(paper 16); over the 10 other seeds ZF6M has 8 to 14 good units, never within 16 +- 1.6. Of the
paper's 100 listed good units, 91 are in the dataset (ZF4F 34, ZF5M 19, ZF6M 16, ZF7F 22) and 70 of
them are good here (29, 14, 9 and 18), so the shortfall in ZF6M comes from listed units that fall
below 12% here, while the 9 listed units absent from the dataset are all of ZF4F.

## fig4b-ensemble-20

> Ensemble decoding performance rises with ensemble size, reaching an average PCC Z+T of 42% for 20
> units, above the 29.6% obtained from response strength alone (Z), a 43% increase.

Ensembles as C7: all single units and pairs, 10,000 random draws for sizes 3, 4, 5, 10, 15 and 20
(code), each decoded once (the Methods: "a single leave-one-out per call-type"; the code ran 100
repetitions, which changes the spread of single-ensemble values but not their mean). C8 t-test
with df 2 neff - 2.

In [ ]:
SIZES = [1, 2, 3, 4, 5, 10, 15, 20]
N_RAND = 10000
ens4b, res4b = {}, {}
seed = 6000
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    cv, nperm, m = readings[name]
    g = good[name]
    rng = np.random.default_rng(6100 if name.startswith("(a)") else 6200)
    for k in SIZES:
        if k <= 2:
            e = [list(c) for c in combinations(g, k)]
        else:
            e = [list(rng.choice(g, k, replace=False)) for _ in range(N_RAND)]
        ens4b[(name, k)] = e
        for cd in ["Z", "ZT"]:
            seed += 1
            res4b[(name, k, cd)] = aa5.run_ensembles(db, e, code=cd, cv=cv, nperm=1, seed=seed)
    curve = pd.DataFrame({cd: [res4b[(name, k, cd)][m].mean() for k in SIZES] for cd in ["Z", "ZT"]}, index=SIZES)
    curve["n_ensembles"] = [len(ens4b[(name, k)]) for k in SIZES]
    print(name, f"({len(g)} good units)")
    print(curve.round(2).T)
    zt, z = res4b[(name, 20, "ZT")][m], res4b[(name, 20, "Z")][m]
    t, df, p = aa5.corrected_ttest(zt, z, len(g), 20)
    print(f"  size 20: Z+T {zt.mean():.2f}, Z {z.mean():.2f}, increase {100 * (zt.mean() / z.mean() - 1):.0f}%; corrected t({df}) = {t:.2f}, p = {p:.2g}")
    print("  Z+T rises at every step:", bool(np.all(np.diff(curve.ZT) > 0)))
print("paper: Z+T 42.3 (tol 4.23), Z 29.6 (tol 2.96), t(160) = 8.83, p < 1e-5")
print(f"elapsed so far {(time.time() - T_START) / 60:.1f} min")

In [ ]:
# Variant: the code's 100 repetitions, on 500 of the 10,000 size-20 ensembles (reading b)
name = "(b) trial, 1 rep, PCC_b"
sub = ens4b[(name, 20)][:500]
v = {cd: aa5.run_ensembles(db, sub, code=cd, cv="trial", nperm=100, seed=6900 + i)["pcc_code"] for i, cd in enumerate(["Z", "ZT"])}
print(f"500 ensembles, 100 repetitions: Z+T {v['ZT'].mean():.2f}, Z {v['Z'].mean():.2f}; one repetition on the same ensembles: "
      f"Z+T {res4b[(name, 20, 'ZT')]['pcc_code'][:500].mean():.2f}, Z {res4b[(name, 20, 'Z')]['pcc_code'][:500].mean():.2f}")
print(f"SD over ensembles, Z+T: 100 reps {v['ZT'].std():.2f}, 1 rep {res4b[(name, 20, 'ZT')]['pcc_code'][:500].std():.2f}")
# Variant: ensembles drawn from the paper's own good units that the dataset holds (code reading)
rng = np.random.default_rng(6950)
e = [list(rng.choice(in_ds, 20, replace=False)) for _ in range(N_RAND)]
vp = {cd: aa5.run_ensembles(db, e, code=cd, cv="trial", nperm=1, seed=6960 + i)["pcc_code"] for i, cd in enumerate(["Z", "ZT"])}
t, df, p = aa5.corrected_ttest(vp["ZT"], vp["Z"], len(in_ds), 20)
print(f"{N_RAND} ensembles of 20 from the paper's {len(in_ds)} listed good units: Z+T {vp['ZT'].mean():.2f}, Z {vp['Z'].mean():.2f}; t({df}) = {t:.2f}, p = {p:.2g}")

In [ ]:
# What the two cross-validations measure, on synthetic ensembles of 20 units with 10 trials of each of the 110
# stimuli: each unit responds with a mean specific to the exemplar (SD 1) and no call-type signal, trial noise SD 1.
rng = np.random.default_rng(6990)
S = len(db["stims"])
syn = {"trial": [], "exemplar": []}
for _ in range(200):
    mu = rng.normal(size=(20, S, 1, 6))
    F = mu + rng.normal(size=(20, S, 10, 6)); Cn = np.full((20, S), 10)
    for cv in syn:
        conf, nt = aa5.decode(F, Cn, db["y"], 10, rng, code="ZT", cv=cv, nperm=1)
        syn[cv].append(aa5.pcc_code(conf, nt))
print("synthetic exemplar-only signal, mean PCC over 200 ensembles: " +
      ", ".join(f"{cv} CV {np.mean(v):.1f}% (SD {np.std(v):.1f})" for cv, v in syn.items()) + "; chance 10%")

**Result: reproduced, under the code's cross-validation, adopted as an assumption.** With one random trial
per stimulus held out (decoders.naiveBayes), the mean PCC of ensembles of 20 is 43.2% with Z+T and
30.4% with Z (paper 42.3% and 29.6%), Z+T rises at every size from 1 to 20, and the corrected t-test
gives t(138) = 9.53, p = 7.5e-17 (138 df from the 89 good units of this reading). Under the Methods'
cross-validation (one exemplar per call type held out) the means are 32.0% and 24.1%, both outside
tolerance, while the rise and the test hold (t(114) = 5.79): partially reproduced under that reading.
100 repetitions on 500 of the ensembles give 43.1% and 30.5%, and ensembles drawn from the paper's own
91 listed good units give 40.7% and 29.4%. The two schemes measure different things: on synthetic
ensembles whose units carry only exemplar-specific responses and no call-type signal, the code's
scheme gives 78.2% and the Methods' 9.8% (chance 10%), because the code's tests trials of exemplars it
was trained on. The outcome rests on the assumption that the paper's values come from the code's
scheme.

## fig4b-top-ensembles

> The top 10% of ensembles of 20 units reach an average PCC Z+T of about 50%, and the best ensembles
> approach the 65% achieved by acoustic classifiers.

The top 10% (Results) or 5% (Methods) of the size-20 ensembles by their first decoding, re-estimated
with a second random held-out draw (Methods).

In [ ]:
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    cv, nperm, m = readings[name]
    first = res4b[(name, 20, "ZT")][m]
    for top in [10, 5]:
        idx = np.argsort(first)[::-1][: int(len(first) * top / 100)]
        sec = aa5.run_ensembles(db, [ens4b[(name, 20)][i] for i in idx], code="ZT", cv=cv, nperm=1, seed=7000 + top + (0 if name.startswith("(a)") else 50))[m]
        print(f"{name}, top {top}% ({len(idx)} ensembles): first draw mean {first[idx].mean():.2f}, re-estimated mean {sec.mean():.2f}, "
              f"best re-estimated {sec.max():.2f}, best first draw {first.max():.2f}")
print("paper: about 50 (tol 5); best close to 65 (read as >= 55)")

**Result: reproduced, under the code's cross-validation (assumption).** The top 10% of the size-20
ensembles, re-estimated with a second held-out draw, average 48.8% (paper about 50%, tolerance 5) and
the top 5% 50.4%. The best re-estimated single draw (86.4%) overstates the best ensemble, as one draw
is noisy (SD over 500 ensembles 7.7 points with one repetition against 4.8 with 100); the best of the
1,800 Figure 5 ensembles decoded with 100 repetitions is 68.4%, above the 55% the criterion reads as
close to 65% (top 10% of them 52.8%). Under the Methods' cross-validation the re-estimated top-10% mean
is 34.0%, outside tolerance, and the best 100-repetition ensemble 41.1%.

## fig4b-confusions

> Most ensemble misclassifications are systematic, Ne confused with Te, LT with Be, Th (Tu) with Te,
> and Di with Ag.

In [ ]:
labels = db["labels"]
pairs = [("Ne", "Te"), ("LT", "Be"), ("Th", "Te"), ("Di", "Ag")]
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    M = aa5.mean_confusion(res4b[(name, 20, "ZT")])
    print(name)
    print(pd.DataFrame(100 * M, index=labels, columns=labels).round(1))
    n_ok = 0
    for a_, b_ in pairs:
        i, j = labels.index(a_), labels.index(b_)
        off = [(M[i, k], labels[k]) for k in range(len(labels)) if k != i]
        order = [l for _, l in sorted(off, reverse=True)]
        rank = order.index(b_) + 1
        n_ok += rank <= 2
        print(f"  true {a_} predicted {b_}: {100 * M[i, j]:.1f}%, rank {rank} of 9 off-diagonal entries (largest: {order[0]} {100 * max(off)[0]:.1f}%)")
    print("  pairs among the two largest of their row:", n_ok, "of 4")

**Result: reproduced.** In the mean row-normalized confusion matrix of the 10,000 ensembles of 20, each
named pair is the largest or second largest off-diagonal entry of its row under both readings. Code
reading: Ne as Te 14.8% (rank 1), LT as Be 11.6% (rank 2), Th as Te 22.5% (rank 1), Di as Ag 10.9%
(rank 2). Methods' reading: ranks 1, 2, 2 and 1.

## figs4-per-bird-and-sex

> Decoding performance and its systematic errors are similar in every bird decoded alone, and PCC does
> not differ significantly between female and male birds (difference 8.3%).

Ensembles of 15 drawn within each bird from its good units: all combinations when there are at most
9,999, otherwise 10,000 random (decodingCallTypesNaiveBayesPart2-SameBird.ipynb, maxTest = 9999).
The test is plotConfusionsBird.ipynb's.

In [ ]:
NE = 15
def per_bird(name, units_, rng, seed0):
    cv, nperm, m = readings[name]
    g = pd.Series(units_); gb = bird_of.loc[g].values
    per = {}
    for i, b in enumerate(sorted(set(bird_of))):
        gu = list(g[gb == b])
        if len(gu) < NE:
            per[b] = dict(ns=len(gu), mean=np.nan, sd=np.nan); continue
        e = [list(c) for c in combinations(gu, NE)] if comb(len(gu), NE) <= 9999 else [list(rng.choice(gu, NE, replace=False)) for _ in range(10000)]
        r = aa5.run_ensembles(db, e, code="ZT", cv=cv, nperm=1, seed=seed0 + i)
        pc = r[m]
        sd = pc.std() * np.sqrt(len(gu) / (len(gu) - NE)) if len(gu) > NE else np.nan
        M = aa5.mean_confusion(r)
        nb = sum(sorted([(M[labels.index(a_), k], labels[k]) for k in range(10) if k != labels.index(a_)], reverse=True).index((M[labels.index(a_), labels.index(b_)], b_)) < 2 for a_, b_ in pairs)
        per[b] = dict(ns=len(gu), n_ens=len(e), mean=pc.mean(), sd=sd, pairs_top2=nb)
    P = pd.DataFrame(per).T
    print(P)
    if P["sd"].notna().all():
        n = P.ns - NE + 1
        def sexstats(b1, b2):
            mean = (n[b1] * P.loc[b1, "mean"] + n[b2] * P.loc[b2, "mean"]) / (n[b1] + n[b2])
            sd = np.sqrt((P.loc[b1, "sd"] ** 2 * (n[b1] - 1) + P.loc[b2, "sd"] ** 2 * (n[b2] - 1)) / (n[b1] + n[b2] - 2))
            return mean, sd
        mf, sf = sexstats("ZF4F", "ZF7F"); mm, sm = sexstats("ZF5M", "ZF6M")
        pooled = np.sqrt((sf ** 2 + sm ** 2) / 2)
        t = (mf - mm) / (pooled * np.sqrt(1 / 2 + 1 / 2))
        print(f"  female {mf:.2f}, male {mm:.2f}, difference {mf - mm:.2f}; t(2) = {t:.3f}, p = {2 * stats.t.sf(abs(t), 2):.3f}")
    else:
        print(f"  birds with fewer than {NE + 1} good units ({', '.join(f'{b} {int(v)}' for b, v in P.ns.items() if v <= NE)}): "
              f"ensembles of {NE} with the corrected SD (sqrt(ns/(ns - {NE}))) need at least {NE + 1}, so the test cannot be run")

for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    print(name, "good units of this attempt")
    per_bird(name, good[name], np.random.default_rng(8000 if name.startswith("(a)") else 8100), 8200 + (0 if name.startswith("(a)") else 10))
print("paper: difference 8.3 (tol 0.83), t(2) = 0.6757, p = 0.5689")

In [ ]:
# Variant: the paper's own listed good units that the dataset holds (paper-good-units-link), within each bird
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    print(name, f"the paper's {len(in_ds)} listed good units held by the dataset")
    per_bird(name, in_ds, np.random.default_rng(8300 if name.startswith("(a)") else 8400), 8500 + (0 if name.startswith("(a)") else 10))

**Result: blocked (other: too few units under the claim's own rule).** Ensembles of 15 with the
corrected SD need at least 16 good units in each bird. ZF6M has 10 good units under the code's reading
(9 under the Methods', where ZF7F also has 14), so the paper's test cannot be run on this attempt's
good units. In the birds with more than 15, mean PCC Z+T at 15 units is 37.9% (ZF4F), 37.4% (ZF5M) and
41.0% (ZF7F) under the code's reading, and 3, 3 and 4 of the four Figure 4B error pairs are among the
two largest entries of their rows. As a variant, on the paper's 91 listed good units the dataset holds
(ZF4F 34, ZF5M 19, ZF6M 16, ZF7F 22), the test runs: female minus male 5.89 points, t(2) = 0.451, p =
0.696 (code reading), and 4.22, t(2) = 0.295, p = 0.796 (Methods' reading), so no sex difference, with
a difference below the paper's 8.3. The link claim's criterion keeps every outcome off the listed
names, so the outcome stays blocked.

## Figure 5: MRFs, volume and the ensemble regressions

1,800 random ensembles of 20 good units, each decoded with 100 held-out repetitions (Methods), under
the two cross-validation readings. MPS of the 110 stimuli (C9, both sound windows); MRFs of each good
unit (six components); LDA axes (C10, both rules); volume, SI, ES and neural noise (C11).

In [ ]:
# The lines of the paper's code (pinned commit) behind details the verifier could not see: ES normalization,
# MRF scaling (one SD pooled over the good units), the LDA axes (eigh), the PCC > 30 filter, the sex test, and the
# neural clustering's threshold grid. allz holds the z of every trial of the good units (the loop over unitNamesGood).
BASE = "https://raw.githubusercontent.com/ftheunissen/Ensemble-Code-Cell-Reports-2022/f2d237639217425e7fc81d694708f19428a3b026/"
want = {"decodingCallTypesNaiveBayesPart2.ipynb": ["pccDiag", "selH[istart"],
        "GenerateMLFilters.ipynb": ["for i,unit in enumerate(unitNamesGood)", "allz.append(", "stdz = ", "stdPC = ", "/(110*stdz)", "np.linalg.eigh", "indGood = (pcc > 30)", "np.linalg.det(np.cov(unitMTFLDA", "unitNoiseComb += std"],
        "plotConfusionsBird.ipynb": ["nE = numUnits[6]", "n0 = 2", "n1 = 2", "tval = ", "fac = np.sqrt"],
        "EPHYS4-Population_Clustering.ipynb": ["delta = (clustered.max()", "start_thresh = clustered.min()", "for cl in range(1,500)"]}
for f, keys in want.items():
    nbc = requests.get(BASE + f, timeout=120).json()
    lines = [ln for c in nbc["cells"] if c["cell_type"] == "code" for ln in "".join(c["source"]).splitlines()]
    print(f)
    for k in keys:
        hit = [ln.strip() for ln in lines if k in ln]
        print(f"   {k!r}: {hit[:2] if hit else 'not found'}")

In [ ]:
t0 = time.time()
mps = {}
for win in ["methods", "code"]:
    out = []
    for s in db["stims"]:
        w = waves[s][0]
        M_, wt, wf = aa5.stimulus_mps(w["wav"], w["tStim0"], w["dt"], win)
        out.append(M_)
    mps[win] = np.array(out)
fsl, tsl = aa5.mps_window(wt, wf)
print(f"MPS of {len(db['stims'])} stimuli x 2 windows in {time.time() - t0:.0f} s; shape {mps['methods'].shape[1:]}, "
      f"kept {fsl.stop - fsl.start} spectral x {tsl.stop - tsl.start} temporal modulations "
      f"({wf[fsl][0] * 1000:.2f} to {wf[fsl][-1] * 1000:.2f} cyc/kHz, {wt[tsl][0]:.1f} to {wt[tsl][-1]:.1f} Hz)")

In [ ]:
N5, NPERM5 = 1800, 100
res5, ens5 = {}, {}
t0 = time.time()
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    cv, nperm, m = readings[name]
    rng = np.random.default_rng(9000 if name.startswith("(a)") else 9100)
    ens5[name] = [list(rng.choice(good[name], 20, replace=False)) for _ in range(N5)]
    res5[name] = aa5.run_ensembles(db, ens5[name], code="ZT", cv=cv, nperm=NPERM5, seed=9200 + (0 if name.startswith("(a)") else 1))
    p = res5[name][m]
    print(f"{name}: {N5} ensembles x {NPERM5} repetitions, mean PCC {p.mean():.2f}, range {p.min():.1f} to {p.max():.1f}, "
          f"above 30%: {(p > 30).sum()} (paper 1746)")
print(f"decoded in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# fig4b-top-ensembles, without the bias of a single draw: the 1,800 ensembles of 20 decoded with 100 repetitions
for name in res5:
    p = res5[name][readings[name][2]]
    top = np.sort(p)[::-1]
    print(f"{name}: best of {len(p)} ensembles {top[0]:.2f}; top 10% mean {top[: len(p) // 10].mean():.2f}, top 5% mean {top[: len(p) // 20].mean():.2f}")

In [ ]:
def unit_regressors(name):
    # Per good unit: MRF projections (per window and LDA rule), SI (a, b), ES (a, b), neural noise (variance, SD)
    g = good[name]
    cv, nperm, m = readings[name]
    allz = np.concatenate([db["trials"][x]["z"].to_numpy() for x in g])
    allT = np.vstack([db["T"][x] for x in g])
    sdz, sdT = allz.std(), allT.std(axis=0)
    sres = single[(name, "ZT")]
    pos = {x: i for i, x in enumerate(full)}
    out = {}
    for x in g:
        tr = db["trials"][x].copy()
        T = db["T"][x]
        for k in range(5):
            tr[f"T{k + 1}"] = T[:, k]
        w = tr.groupby("stim")[["z", "T1", "T2", "T3", "T4", "T5"]].mean().reindex(db["stims"])
        za, zb = aa5.zct_unit(byuid[x], db["trials"][x])
        noise_var = np.mean([g_.z.var(ddof=0) for _, g_ in tr.groupby("call")])
        noise_sd = np.mean([g_.z.std(ddof=0) for _, g_ in tr.groupby("call")])
        conf, nt = sres["conf"][pos[x]], sres["ntest"][pos[x]]
        out[x] = dict(w=w.to_numpy(), SI_a=aa5.selectivity_index(za.values()), SI_b=aa5.selectivity_index(zb.values()),
                      ES_a=aa5.entropy_selectivity(conf, nt, True), ES_b=aa5.entropy_selectivity(conf, nt, False),
                      noise_var=noise_var, noise_sd=noise_sd)
    scale = np.concatenate([[sdz], sdT])
    proj = {}
    for win in ["methods", "code"]:
        X = mps[win][:, fsl, tsl]
        dev = (X - X.mean(axis=0)).reshape(len(X), -1)
        for rule in ["largest", "eigh"]:
            axes, _ = aa5.lda_axes(X, db["y"], rule)
            for x in g:
                mrf = dev.T @ np.nan_to_num(out[x]["w"])            # (features, 6): sum over stimuli
                proj[(win, rule, x)] = (axes.T @ mrf).T / (110 * scale[:, None])   # 6 x 3
    return out, proj

def ensemble_table(name, out, proj):
    rows = []
    for e in ens5[name]:
        r = {}
        for win in ["methods", "code"]:
            for rule in ["largest", "eigh"]:
                V = np.vstack([proj[(win, rule, x)] for x in e])
                r[f"logvol_{win}_{rule}"] = np.log(np.linalg.det(np.cov(V.T)))
        for k in ["SI_a", "SI_b", "ES_a", "ES_b", "noise_var", "noise_sd"]:
            r[k] = np.mean([out[x][k] for x in e])
        rows.append(r)
    return pd.DataFrame(rows)

t0 = time.time()
tab5 = {}
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    out, proj = unit_regressors(name)
    tab5[name] = ensemble_table(name, out, proj)
    tab5[name]["pcc"] = res5[name][readings[name][2]]
print(f"regressors in {time.time() - t0:.0f} s")
print(tab5["(b) trial, 1 rep, PCC_b"].describe().T.round(3))

def regress(name, xcols, filt):
    d = tab5[name]
    if filt == "pcc > 30":
        d = d[d.pcc > 30]
    return aa5.corrected_regression(d[xcols].to_numpy(), d.pcc.to_numpy(), len(good[name]), 20)

### fig5e-volume

> Across random ensembles of 20 units, PCC increases with the volume the ensemble's MRF components span
> in the 3D LDA space of call types (R2adj = 0.1).

In [ ]:
rows = []
for name in tab5:
    for filt in ["pcc > 30", "all"]:
        for win in ["methods", "code"]:
            for rule in ["largest", "eigh"]:
                r = regress(name, [f"logvol_{win}_{rule}"], filt)
                rows.append(dict(cv=name[:3], ensembles=filt, n=r["n"], window=win, lda=rule, slope=r["coef"][1], R2adj=r["r2adj"],
                                 F=r["F"], df=str(r["df"]), p=r["p"]))
fig5e = pd.DataFrame(rows)
print(fig5e.round(4).to_string(index=False))
print("paper: R2adj 0.1 (tol 0.01), F(1,79) = 8.975, p = 0.0036, 1746 ensembles")

**Result: reproduced, under the code's cross-validation, sound window and LDA axes (assumption).** PCC
rises with log volume under every reading, with corrected p of 0.011 or less. With the code's choices
throughout (one random trial held out, the -500 to 500 ms window and numpy eigh's first three axes),
R2adj is 0.094 over the 1,797 ensembles with PCC above 30% (F(1,68) = 7.28, p = 0.0088), within 0.1 +-
0.01; the criterion names the window and axes as readings, and these are the choices that made the
paper's printed F(1,79) = 8.975. With the Methods' window and the axes of largest eigenvalue R2adj is
0.120 (F(1,68) = 9.60, p = 0.0028), outside tolerance. Under the Methods' cross-validation R2adj is
0.11 to 0.30. Here 1,797 of the 1,800 ensembles exceed 30%, against the paper's 1,746.

### fig5f-si-unrelated

> The average Z-based selectivity index of the units in an ensemble is not related to the ensemble's
> decoding performance.

In [ ]:
rows = []
for name in tab5:
    for filt in ["pcc > 30", "all"]:
        for si in ["SI_a", "SI_b"]:
            r = regress(name, [si], filt)
            rows.append(dict(cv=name[:3], ensembles=filt, n=r["n"], SI=si, slope=r["coef"][1], R2adj=r["r2adj"], F=r["F"], df=str(r["df"]), p=r["p"]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper (code output): R2adj = -0.0006, p = 0.99")

**Result: reproduced.** The ensemble-corrected F-test of PCC on the ensemble mean of the Methods' SI gives
p = 0.85 (R2adj -0.0001) under the code's cross-validation and p = 0.32 under the Methods'; the code's
SI gives p = 0.67 and 0.30. No reading finds a relation at p < 0.05.

### fig5g-entropy-selectivity

> The ensemble average Entropy Selectivity of its units is negatively correlated with ensemble decoding
> performance.

In [ ]:
rows = []
for name in tab5:
    for filt in ["pcc > 30", "all"]:
        for es in ["ES_a", "ES_b"]:
            r = regress(name, [es], filt)
            rows.append(dict(cv=name[:3], ensembles=filt, n=r["n"], ES=es, slope=r["coef"][1], R2adj=r["r2adj"], F=r["F"], df=str(r["df"]), p=r["p"]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: negative, F(1,79) = 6.73, p = 0.011 (code R2adj 0.076)")

**Result: reproduced.** With the Methods' ES formula (reading b), the slope is negative and significant
under both cross-validations (code: -69.4, F(1,68) = 4.87, p = 0.031; Methods: -50.0, p = 0.0005, over
the ensembles above 30%). With the code's normalized ES (reading a) the slope is negative under the
Methods' cross-validation (p = 0.046) but positive and not significant under the code's (p = 0.79). The Methods' ES takes negative values here (ensemble means -0.18 to -0.07), while the Figure 5G
legend says ES ranges from 0 to 1, which only the code's normalized ES does (0.034 to 0.095); whether
reading (b) belongs in the criterion is asked of a reader (questions/robotka-2023.yaml).

### fig5h-neural-noise

> As neural variability within call types (neural noise plus lack of invariance) increases, ensemble
> decoding performance decreases (R2adj = 0.1).

In [ ]:
rows = []
for name in tab5:
    for filt in ["pcc > 30", "all"]:
        for nz in ["noise_var", "noise_sd"]:
            r = regress(name, [nz], filt)
            rows.append(dict(cv=name[:3], ensembles=filt, n=r["n"], noise=nz, slope=r["coef"][1], R2adj=r["r2adj"], F=r["F"], df=str(r["df"]), p=r["p"]))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: negative, R2adj 0.1 (tol 0.01), F(1,79) = 8.75, p = 0.004")

**Result: partially reproduced.** PCC falls with the ensemble mean neural noise under every reading (code
cross-validation, variance: slope -139, F(1,68) = 18.6, p = 0.0001), but R2adj is 0.21 with the
variance and 0.13 with the SD under the code's cross-validation, and 0.15 to 0.27 under the Methods',
all outside 0.1 +- 0.01.

### fig5-multiple-regression

> A linear model of PCC on volume spanned, neural variability and the two selectivity measures together
> explains R2adj = 0.22 of the variance of ensemble decoding performance.

In [ ]:
rows = []
for name in tab5:
    for filt in ["pcc > 30", "all"]:
        for win in ["methods", "code"]:
            for rule in ["largest", "eigh"]:
                r = regress(name, ["noise_var", f"logvol_{win}_{rule}", "ES_a", "SI_b"], filt)
                r2 = regress(name, ["noise_var", f"logvol_{win}_{rule}", "ES_a", "SI_a"], filt)
                rows.append(dict(cv=name[:3], ensembles=filt, n=r["n"], window=win, lda=rule, R2adj_SIb=r["r2adj"], F_SIb=r["F"],
                                 p_SIb=r["p"], R2adj_SIa=r2["r2adj"], p_SIa=r2["p"], df=str(r["df"])))
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: R2adj 0.22 (tol 0.022), F(4,76) = 5.832, p = 0.00038")
print(f"total run time {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (this process)")

**Result: reproduced, under the code's cross-validation and LDA axes (assumption).** With numpy eigh's
axes, as the code, the four regressors give R2adj = 0.238 (Methods' window) and 0.241 (code's window)
over the ensembles above 30%, F(4,65) = 5.49 and 5.59, p = 0.0007 and 0.0006, within 0.22 +- 0.022.
With the axes of largest eigenvalue R2adj is 0.252 and 0.248, just outside tolerance. Under the
Methods' cross-validation R2adj is 0.30 to 0.49.

## fig6c-number-of-clusters (call-type half)

> The optimal number of neural clusters varies from 1 to 30 across ensembles, but its mean and median are
> close to the number of call types whether call types (mean 10.83, median 9) or acoustic labels (mean
> 10.03, median 9) are the reference, with no significant difference between the two.

The claim is blocked: the acoustic labels are not public. Its call-type values need only this dataset, and
the criterion judges them on their own, so they are computed here. Neural clustering as C12 (aa5.run_clustering)
on the 1,800 Figure 5 ensembles of each reading of the good units (the criterion's variant; the legend's
10,000 were not run, for time), (a) UMAP then Ward (the code), (b) Ward on the averaged responses. The tree is cut at 499 thresholds
between its smallest and largest linkage distance, as C12 states; as a variant, between the minimum and
maximum of the whole linkage matrix, its index and count columns included, as the code computes them
(`clustered.min()` and `clustered.max()`, printed in the code-lines cell above). Both cuts are made on the
same trees.

In [ ]:
import os
os.environ.setdefault("NUMBA_NUM_THREADS", "1")
t0 = time.time()
rows = []
for name in ["(a) exemplar, PCC_a", "(b) trial, 1 rep, PCC_b"]:
    for use_umap in [True, False]:
        ari_d, k_d, ari_m, k_m = aa5.run_clustering(db, ens5[name], use_umap=use_umap, seed=9500 + (0 if name.startswith("(a)") else 10) + use_umap)
        for grid, ari, k in [("distances (C12)", ari_d, k_d), ("matrix (code)", ari_m, k_m)]:
            for filt, sel in [("all", np.ones(len(k), bool)), ("pcc > 30", res5[name][readings[name][2]] > 30)]:
                rows.append(dict(units=name[:3], clustering="(a) UMAP + Ward" if use_umap else "(b) Ward", grid=grid, ensembles=filt, n=int(sel.sum()),
                                 mean_k=k[sel].mean(), median_k=np.median(k[sel]), min_k=k[sel].min(), max_k=k[sel].max(), mean_ARIcall=ari[sel].mean()))
print(pd.DataFrame(rows).round(3).to_string(index=False))
# grid: thresholds between the smallest and largest linkage distance (C12), or between the min and max of the whole
# linkage matrix, index and count columns included (the code, printed above), a variant
print("paper, call types: mean 10.83 (tol 1.083), median 9 (tol 0.9), range 1 to 30")
print(f"clustered in {(time.time() - t0) / 60:.1f} min")

**Result: blocked (data missing: the acoustic labels).** The call-type half alone: with UMAP then Ward
(the code) and the tree cut between its smallest and largest linkage distance (C12), the optimal number
of clusters over the 1,800 ensembles of the code reading's good units has mean 10.4 and median 10
(range 2 to 33), and over those of the Methods reading's good units mean 10.3 and median 10 (range 2
to 30); the paper gives mean 10.83 and median 9 (range 1 to 30). The mean is within tolerance in both,
the median in neither. With the code's threshold grid (minimum and maximum over the whole linkage
matrix) on the same trees, the means are 9.9 and 9.8 and the medians 9, all within tolerance. Ward on
the averaged responses gives means of 11.5 to 11.9 and medians of 11 under both grids. No ensemble's
best cut has 1 cluster (minimum 2). The acoustic half and the paired test cannot be run.

## Summary

| claim | outcome |
|---|---|
| stimulus-set-link | reproduced |
| paper-good-units-link | partially reproduced (91 of 100 units found) |
| auditory-units | reproduced |
| inhibited-units | reproduced |
| table1-snr | reproduced |
| pcs-variance | reproduced |
| fig3c-peak-call-types | reproduced (assumption: the code's zCT) |
| fig4a-unit-z-vs-zt | reproduced |
| good-units-100 | not reproduced (89 against 100 +- 10; 87 to 101 over seeds) |
| fig4b-ensemble-20 | reproduced (assumption: the code's cross-validation) |
| fig4b-top-ensembles | reproduced (same assumption) |
| fig4b-confusions | reproduced |
| figs4-per-bird-and-sex | blocked: ZF6M has fewer than 16 good units (the paper's listed units, a variant, give p = 0.70) |
| fig5e-volume | reproduced (assumption: the code's cross-validation, window and LDA axes) |
| fig5f-si-unrelated | reproduced |
| fig5g-entropy-selectivity | reproduced |
| fig5h-neural-noise | partially reproduced |
| fig5-multiple-regression | reproduced (assumption: the code's cross-validation and LDA axes) |
| figs6, figs5-region-anova, figs9 | blocked: no brain area per unit |
| fig6a, acoustic-labels-decoding, fig6c (2), figs10 | blocked: acoustic database not public (the call-type half of fig6c-number-of-clusters is computed: with the C12 threshold grid, mean 10.4 and median 10 on the code reading's good units, mean 10.3 and median 10 on the Methods reading's; with the code's grid, means 9.9 and 9.8, medians 9) |
| anesthetized claims (4) | blocked: CRCNS aa-4 not analysed in this run |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The Methods and the paper's code differ on the
decoder's cross-validation (one exemplar per call type held out, against one random trial per
stimulus); both are run, and the ensemble claims are judged on the code's, whose results match the
paper's printed values (an assumption). The code's scheme tests trials of exemplars it was trained on,
so it rewards exemplar-specific responses as well as call-type ones (78% against 10% on synthetic
units with exemplar signal alone). For Figure 5 the code's sound window and LDA axes are likewise
adopted where they decide the outcome, as the paper's printed values come from them. The Figure 4B ensembles are decoded once each (the Methods'
single leave-one-out; the code ran 100 repetitions). Both readings of zCT, SI, ES, neural noise,
the MPS window, the LDA axes and the 30% ensemble filter are shown. The good-unit set is selected here
(89 units) and differs from the paper's 100. The best ensemble of fig4b-top-ensembles is taken from the
1,800 Figure 5 ensembles decoded with 100 repetitions, not from the Figure 4B ensembles decoded once.
The neural clustering of Figure 6C is run on the 1,800 Figure 5 ensembles, not the legend's 10,000,
and its tree is cut on both the C12 threshold grid and the code's.

**What was found about the dataset.** The files match the paper's stimulus set exactly and hold 4
birds, 50 sites and 872 unit pickles, 34 of them empty and 112 with song stimuli only. Unit counts are
close to the paper's (726 units with vocalization trials against 730; 287 auditory against 292; 187
with all 110 stimuli against 191). Nine of the paper's 100 listed good units come from a site sorting
named ZF4F_2t_190613_101015.old that is not in the dataset, whose site of that name holds other
clusters. No unit carries a brain area, so the claims by area cannot be tested. KDE_BW (0.2 to 10, 646
of the 872 pickles at 10) is, as the descriptor says, the bandwidth of the coarse rate used for the
rate threshold, not that of spikeKDE. The stored spikeKDE is a 30 ms
Gaussian KDE of the spike times, as the paper says, but scaled by the number of trials over 5 (ratio
1.00 at 5 trials, 2.01 at 10, 3.01 at 15), so it is not the per-trial rate the descriptor's
"time-varying rate" suggests. Each tar holds an AppleDouble
`._` file for every pickle.